| 제목 | 내용 | 설명 |
| :--- | :--- | :--- |
| contracts.py | 시스템 최하단의 기준표 모듈 | 허용 및 금지 툴 목록, 툴별 필수 및 선택 인자 스펙을 정의하는 단일 진실 공급원입니다. |
| validation.py | 규칙 기반 툴 플랜 정적 및 동적 검증 모듈 | 입력된 툴 플랜의 최상위 구조 스키마와 인자의 유효성, 이유 필드의 품질을 검사하여 최종 상태를 판정합니다. |
| tool_executor.py | 검증 및 디스패치 기반의 툴 실행 계층 | 실시간 런타임 환경에서 평가용 검증 로직을 중립화하여 코드를 재사용하고 안전하게 분기 실행합니다. |
| mcp_server.py | MCP 서버 통합 진입점 | 외부 통신을 담당하는 얇은 계층으로 외부 SDK 설치 여부와 무관하게 순수 Python 함수형 골격으로 동작합니다. |

```mermaid
graph TD
    %% 노드 정의 (특수기호 및 HTML 태그 배제)
    User((사용자 질의))
    MCP[mcp_server.py - 통합 진입점]
    Selector[tool_selector.py]
    Prompt[프롬프트 조립]
    LLM{LLM API 호출}
    RawPlan[Raw Tool Plan]
    Fallback[규칙 기반 Fallback]
    
    Executor[tool_executor.py]
    Normalizer[normalizer.py - 구조 사전 보정]
    N_Details[인자 제거 및 엔티티 기반 보강]
    Validator[validation.py - 안전 및 계약 검증]
    V_Details[스키마 검사 및 금지 툴 차단]
    Gate{상태 게이트키핑}
    
    Blocked[실행 차단 - Rejected]
    NeedsReview[기본 미실행 - Needs Review]
    ExecuteReal[실제 Client 함수 호출]
    Sanitize[결과 마스킹 및 위생화]
    Output((최종 결과 반환))

    %% 파이프라인 흐름 정의
    User --> MCP
    
    MCP -->|1. Selection 단계| Selector
    
    subgraph Selection Phase
        Selector --> Prompt
        Prompt --> LLM
        LLM -->|성공: JSON 반환| RawPlan
        LLM -->|실패 또는 불가| Fallback
        Fallback --> RawPlan
    end

    MCP -->|2. Execution 단계| Executor
    RawPlan --> Executor

    subgraph Execution Phase
        Executor --> Normalizer
        Normalizer --> N_Details
        N_Details --> Validator
        Validator --> V_Details
        V_Details --> Gate
        
        Gate -->|FAIL 또는 NEEDS_REVIEW| Blocked
        Gate -->|WARNING| NeedsReview
        Gate -->|PASS| ExecuteReal
        
        ExecuteReal --> Sanitize
    end

    Sanitize --> Output
    Blocked --> Output
    NeedsReview --> Output


```mermaid
graph TD
    %% 노드 정의
    User((사용자 질의))
    Server[my_mcp_server.py - 진입점]
    Logger[my_mcp_logging.py - 이력 기록]
    
    %% Selection
    subgraph 1. Selection Phase
        Selector[my_tool_selector.py]
        LLM{LLM API 호출}
        Fallback[Rule Fallback - 4개 크롤러 전부 선택]
    end
    
    %% Execution
    subgraph 2. Execution Phase
        Executor[my_tool_executor.py]
        Validator[내부 검증 - 허용 Tool 및 필수 인자]
    end
    
    %% Crawling
    subgraph 3. Crawling Phase
        Crawler[my_crawlers.py]
        Source[(arXiv API / RSS / Google News)]
    end
    
    %% Post-processing
    subgraph 4. Post-processing Phase
        Reranker[my_reranking.py - 관련성 재정렬]
        Security[my_security.py - 민감정보 마스킹]
    end
    
    Output((최종 결과 반환))

    %% 흐름 연결
    User --> Server
    Server -->|1. 계획 생성 요청| Selector
    
    Selector --> LLM
    LLM -->|성공: JSON Plan 반환| Executor
    LLM -->|실패 또는 불가| Fallback
    Fallback -->|안전망 Plan 반환| Executor
    
    Executor --> Validator
    Validator -->|FAIL: 차단| Output
    Validator -->|PASS: 실행| Crawler
    
    Crawler <-->|HTTP/RSS 데이터 수집| Source
    
    Crawler -->|Raw 크롤링 결과| Reranker
    Reranker -->|점수 기반 정렬 완료| Security
    Security -->|안전한 결과물| Executor
    
    Executor -->|최종 실행 집계| Server
    Server -.->|주요 이벤트| Logger
    Server --> Output

```mermaid
flowchart TD
    User([사용자 쿼리]) --> Server[my_mcp_server.py\ncall_execute_query]

    Server --> Selector[my_tool_selector.py\nselect_tool_plan_with_llm]
    Selector -->|LLM 성공| LLM[LLM Tool Plan]
    Selector -->|LLM 실패| Rule[Rule Fallback Plan\n5개 Tool 전부]
    LLM --> Plan{{Tool Plan\nplan: step 1~5}}
    Rule --> Plan

    Plan --> Executor[my_tool_executor.py\nexecute_tool_plan]
    Executor -->|FAIL| Rejected([rejected])
    Executor -->|PASS / WARNING| Run[execute_single_tool × N]

    Run --> C1[search_reddit\nGoogle News RSS]
    Run --> C2[search_arxiv\narXiv Atom API]
    Run --> C3[search_robot_news\nRobohub / Robot Report RSS]
    Run --> C4[search_irobotnews\n로봇신문 Google News RSS]
    Run --> C5[search_robot_db\nLanceDB 캐시 조회]

    C5 --> Lance[my_lance_db.py\nD:/lance_db]
    Lance --> Repo[my_repositories.py\nrobot_multi_source_matrix]
    Lance -->|미설치/경로 없음| LanceFallback([빈 list fallback])

    C1 & C2 & C3 & C4 & Repo --> Rerank[my_reranking.py\nrerank_crawler_results\ntitle+3 / preview+2 / 소스선호+2]
    Rerank --> Sanitize[my_security.py\nsanitize_result\nwhitelist 필드만 통과]
    Sanitize --> Log[my_mcp_logging.py\nlog_event]
    Log --> Result([결과 반환\noriginal_rank / rerank_score / rerank_reasons 포함])

    Executor -.->|검증 기준| Contracts[my_contracts.py\nALLOWED_TOOLS\nTOOL_CONTRACTS]
